# 02 – Compare schemas across the three TNP datasets

**Goal:** the portal splits trip data into three datasets (2018–2022, 2023–2024, 2025 onward).
Before writing `ingest`, we need to know exactly how their schemas differ, because ingest has to
map all three into one interim schema.

This notebook checks, for each dataset:

1. whether all of its files share one header,
2. which columns it has, and which are missing compared with the others,
3. whether column names differ only in spelling or case,
4. the inferred type and null rate of each column,
5. the *format* of the raw strings (e.g. `$12.50` vs `12.50`), which can differ even when
   the column name and type match.

**Before running:** put each dataset's files in its own folder under `data/raw/`, named by the
portal's dataset ID. The ingest stage can use the same layout to record where each row came from.

```
data/raw/m6dm-c72p/   2018–2022 (e.g. the 2021 day)
data/raw/n26f-ihde/   2023–2024 (the Nov 2023 and Mar 2024 samples)
data/raw/6dvr-xwnh/   2025 onward (the Sep 2026 sample)
```

## Setup

In [ ]:
from pathlib import Path

import duckdb
import polars as pl

from rideforecast.config import load_config


def find_project_root(start: Path) -> Path:
    """Walk upward from `start` until a directory containing pyproject.toml is found."""
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError(f"No pyproject.toml found at or above {start}")


ROOT = find_project_root(Path.cwd())
cfg = load_config(ROOT / "configs" / "base.yaml")
RAW_DIR = ROOT / cfg.paths.raw

pl.Config.set_tbl_rows(100)
pl.Config.set_fmt_str_lengths(60)
pl.Config.set_tbl_width_chars(200)

con = duckdb.connect()

`SOURCES` maps a readable label to the portal dataset ID, which is also the folder name.
Each dataset gets an all-`VARCHAR` view, so later checks see the raw strings.
`union_by_name` lines up columns by name if files within one dataset have different headers;
section 1 checks whether that actually happens.

In [ ]:
SOURCES = {
    "2018-2022": "m6dm-c72p",
    "2023-2024": "n26f-ihde",
    "2025-": "6dvr-xwnh",
}
LABELS = list(SOURCES)

files_by_dataset: dict[str, list[str]] = {}
for label, dataset_id in SOURCES.items():
    files = sorted((RAW_DIR / dataset_id).glob("*.csv.gz"))
    assert files, f"No CSV files for {label} in {RAW_DIR / dataset_id}"
    files_by_dataset[label] = [f.as_posix() for f in files]
    print(f"{label} ({dataset_id}): {len(files)} file(s)")
    for f in files:
        print(f"    {f.name}")


def read_csv_sql(label: str, **options: str) -> str:
    """Build a DuckDB read_csv(...) call over every file of one dataset."""
    file_list = ", ".join(f"'{f}'" for f in files_by_dataset[label])
    extra = "".join(f", {key} = {value}" for key, value in options.items())
    return f"read_csv([{file_list}], union_by_name = true{extra})"


VIEWS = {label: f"raw_{dataset_id.replace('-', '_')}" for label, dataset_id in SOURCES.items()}
for label, view in VIEWS.items():
    con.sql(
        f"CREATE OR REPLACE VIEW {view} AS SELECT * FROM {read_csv_sql(label, all_varchar='true')}"
    )

## 1. Does each dataset have one consistent header?

If a dataset's files have different headers, that's a schema change *within* a dataset,
and ingest needs to handle it per file rather than per dataset.

In [ ]:
header_rows = []
for label, files in files_by_dataset.items():
    for f in files:
        cols = con.sql(f"SELECT * FROM read_csv('{f}', all_varchar = true) LIMIT 0").columns
        header_rows.append(
            {
                "dataset": label,
                "file": Path(f).name,
                "n_columns": len(cols),
                "header": "|".join(cols),
            }
        )

headers = pl.DataFrame(header_rows)
headers.group_by("dataset", maintain_order=True).agg(
    pl.len().alias("files"),
    pl.col("header").n_unique().alias("distinct_headers"),
    pl.col("n_columns").unique().alias("column_counts"),
)

### Size and date range of each sample

A quick sanity check that each folder holds the period you think it does.
The start-timestamp column is found by name, in case its exact spelling differs between datasets.

In [ ]:
TS_FORMAT = "%m/%d/%Y %I:%M:%S %p"

overview_rows = []
for label, view in VIEWS.items():
    cols = con.sql(f"SELECT * FROM {view}").columns
    start_col = next(c for c in cols if "start" in c.lower() and "timestamp" in c.lower())
    parsed = f"try_strptime(\"{start_col}\", '{TS_FORMAT}')"
    row = con.sql(f"""
        SELECT
            '{label}' AS dataset,
            count(*) AS n_rows,
            count(*) - count({parsed}) AS unparsed_start,
            min({parsed}) AS first_start,
            max({parsed}) AS last_start
        FROM {view}
    """).pl()
    overview_rows.append(row)

pl.concat(overview_rows)

## 2. Which columns does each dataset have?

`True` means the column exists in that dataset. Columns are listed in order of first
appearance. The portal notes that the 2023–2024 dataset added three new columns, so expect at
least three rows where `2018-2022` is `False`.

In [ ]:
columns_by_dataset = {label: con.sql(f"SELECT * FROM {VIEWS[label]}").columns for label in LABELS}
all_columns = list(dict.fromkeys(c for cols in columns_by_dataset.values() for c in cols))

presence = pl.DataFrame(
    {
        "column": all_columns,
        **{label: [c in cols for c in all_columns] for label, cols in columns_by_dataset.items()},
    }
).with_columns(pl.all_horizontal(LABELS).alias("in_all"))

print(
    f"{len(all_columns)} distinct column names; {presence['in_all'].sum()} present in all datasets"
)
presence

In [ ]:
# Only the columns that are missing from at least one dataset
presence.filter(~pl.col("in_all"))

### Column order

Ingest should always select columns by name, so order differences are harmless as long as
nobody reads these files by position. Listed here so you know whether that's a live risk.

In [ ]:
common = [c for c in all_columns if all(c in cols for cols in columns_by_dataset.values())]
positions = pl.DataFrame(
    {
        "column": common,
        **{label: [cols.index(c) for c in common] for label, cols in columns_by_dataset.items()},
    }
)
moved = positions.filter(pl.concat_list(LABELS).list.n_unique() > 1)
print("Same order everywhere." if moved.is_empty() else f"{moved.height} columns change position:")
moved

## 3. Inferred types and null rates

One full-scan `SUMMARIZE` per dataset. A column with different inferred types across datasets
(e.g. `BIGINT` in one, `VARCHAR` in another) usually means the raw format differs, which
section 4 explains. Different null rates can mean a reporting change, or just that the samples
cover different times of day.

In [ ]:
summaries = []
for label in LABELS:
    s = con.sql(f"SUMMARIZE SELECT * FROM {read_csv_sql(label, sample_size='-1')}").pl()
    summaries.append(
        s.select(
            "column_name",
            "column_type",
            pl.col("null_percentage").cast(pl.Float64),
        ).with_columns(pl.lit(label).alias("dataset"))
    )
summary = pl.concat(summaries)

types_wide = summary.pivot(on="dataset", index="column_name", values="column_type").with_columns(
    (pl.concat_list(LABELS).list.drop_nulls().list.n_unique() > 1).alias("type_differs")
)
types_wide

In [ ]:
nulls_wide = summary.pivot(on="dataset", index="column_name", values="null_percentage")
nulls_wide

## 4. Raw value formats

Types can match while formats don't. For every non-null value we build a *pattern*: each run
of digits becomes `9` and each run of letters becomes `a`. So `$12.50` → `$9.9`,
`1,234` → `9,9`, and `11/05/2023 01:45:00 AM` → `9/9/9 9:9:9 a`.

The table shows the three most common patterns per column and dataset, with a real example
and the share of non-null values that follow each pattern.

In [ ]:
PATTERN_SQL = """
WITH long AS (
    UNPIVOT (SELECT * FROM {view}) ON COLUMNS(*) INTO NAME column_name VALUE value
),
patterns AS (
    SELECT
        column_name,
        left(regexp_replace(regexp_replace(value, '[0-9]+', '9', 'g'), '[A-Za-z]+', 'a', 'g'), 40)
            AS pattern,
        count(*) AS n,
        any_value(value) AS example
    FROM long
    GROUP BY ALL
)
SELECT
    '{label}' AS dataset,
    column_name,
    row_number() OVER (PARTITION BY column_name ORDER BY n DESC) AS rank,
    pattern,
    round(100 * n / sum(n) OVER (PARTITION BY column_name), 2) AS pct,
    example
FROM patterns
"""

all_patterns = pl.concat(
    [con.sql(PATTERN_SQL.format(view=VIEWS[label], label=label)).pl() for label in LABELS]
).sort("column_name", "dataset", "rank")

# The three most common patterns per column and dataset, for reading
patterns = all_patterns.filter(pl.col("rank") <= 3)
patterns

### Which formats appear in one dataset but not another?

Comparing only the *most common* pattern is misleading: if one sample has more long trips,
`9,9` can overtake `9` in Trip Seconds even though every dataset formats numbers the same way.
Shares shift with the sample; what matters is whether a format **exists** in one dataset and
**never appears** in another.

So for each column, take every pattern that makes up at least `MIN_PCT` percent of values in
some dataset, and list the datasets (that have the column) where that pattern never occurs.
An empty table means the formats agree everywhere.

ID columns such as `Trip ID` can show up here because hashed IDs produce many patterns by
chance; ignore them.

In [ ]:
MIN_PCT = 1.0

exists = pl.DataFrame(
    [
        {"column_name": c, "dataset": label}
        for label, cols in columns_by_dataset.items()
        for c in cols
    ]
)
common_patterns = (
    all_patterns.group_by("column_name", "pattern")
    .agg(pl.col("pct").max().alias("max_pct_elsewhere"))
    .filter(pl.col("max_pct_elsewhere") >= MIN_PCT)
)
format_gaps = (
    common_patterns.join(exists, on="column_name")
    .join(
        all_patterns.select("column_name", "pattern", "dataset", "pct"),
        on=["column_name", "pattern", "dataset"],
        how="left",
    )
    .filter(pl.col("pct").is_null())
    .select("column_name", "pattern", "max_pct_elsewhere", pl.col("dataset").alias("missing_in"))
    .sort("column_name", "pattern", "missing_in")
)

differing = format_gaps["column_name"].unique().sort().to_list()
print("Formats agree in every column." if not differing else f"Formats differ in: {differing}")
format_gaps

In [ ]:
# Detail for any single column, e.g. to confirm the '$' and ',' findings from notebook 01
COLUMN = "Fare"
patterns.filter(pl.col("column_name") == COLUMN)